In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 400

# --- Customer categories ---
categories = np.random.choice(
    ["early_career_urban", "platform_worker"],
    size=n,
    p=[0.5, 0.5]
)

# --- Initialize arrays ---
income = np.zeros(n)
debt_to_income = np.zeros(n)
missed_payments_12m = np.zeros(n)
credit_utilization = np.zeros(n)
months_with_company = np.zeros(n)

# --- Generate features by category ---
for i, cat in enumerate(categories):
    
    if cat == "early_career_urban":
        # Lower income, shorter history, more behavioral signals
        income[i] = np.random.normal(2500, 600)
        debt_to_income[i] = np.random.normal(0.45, 0.15)
        missed_payments_12m[i] = np.random.poisson(1.5)
        credit_utilization[i] = np.random.beta(2, 2)  # more spread
        months_with_company[i] = np.random.randint(3, 36)
        
    else:  # platform_worker
        # More variability, stronger financial stress signals
        income[i] = np.random.normal(3000, 1200)
        debt_to_income[i] = np.random.normal(0.6, 0.2)
        missed_payments_12m[i] = np.random.poisson(1.0)
        credit_utilization[i] = np.random.beta(3, 1.5)  # skewed high
        months_with_company[i] = np.random.randint(1, 48)

# --- Clip values to realistic ranges ---
income = np.clip(income, 800, 10000)
debt_to_income = np.clip(debt_to_income, 0, 1.5)
credit_utilization = np.clip(credit_utilization, 0, 1)
missed_payments_12m = np.clip(missed_payments_12m, 0, 10)

# --- Generate default probability using different mechanisms ---
logit = np.zeros(n)

for i, cat in enumerate(categories):
    
    if cat == "early_career_urban":
        # Behavioral risk dominates
        logit[i] = (
            -2.5
            + 1.2 * missed_payments_12m[i]
            + 2.5 * credit_utilization[i]
            + 0.5 * debt_to_income[i]
            - 0.0003 * income[i]
            - 0.02 * months_with_company[i]
        )
        
    else:  # platform_worker
        # Financial instability dominates
        logit[i] = (
            -2.0
            + 3.0 * debt_to_income[i]
            + 1.5 * credit_utilization[i]
            + 0.3 * missed_payments_12m[i]
            - 0.0002 * income[i]
            - 0.015 * months_with_company[i]
        )

# --- Convert to probability ---
prob = 1 / (1 + np.exp(-logit))

# --- Sample defaults ---
default = (np.random.rand(n) < prob).astype(int)

# --- Create DataFrame ---
df = pd.DataFrame({
    "customer_category": categories,
    "income": income,
    "debt_to_income": debt_to_income,
    "missed_payments_12m": missed_payments_12m,
    "credit_utilization": credit_utilization,
    "months_with_company": months_with_company,
    "default": default
})

# --- Optional: quick sanity check ---
print(df.head())
print("\nDefault rate by category:")
print(df.groupby("customer_category")["default"].mean())
# Save

df.to_csv("credit_default.csv", index=False)

# Preview
df.head()

    customer_category       income  debt_to_income  missed_payments_12m  \
0  early_career_urban  3283.287284        0.453151                  2.0   
1     platform_worker  1976.088070        0.597144                  0.0   
2     platform_worker  3299.177450        0.664904                  4.0   
3     platform_worker  2539.814969        0.505792                  0.0   
4  early_career_urban  1655.521735        0.342233                  2.0   

   credit_utilization  months_with_company  default  
0            0.574281                 21.0        0  
1            0.855863                 39.0        1  
2            0.969859                 27.0        1  
3            0.939466                  7.0        0  
4            0.461253                 30.0        1  

Default rate by category:
customer_category
early_career_urban    0.414508
platform_worker       0.507246
Name: default, dtype: float64


,customer_category,income,debt_to_income,missed_payments_12m,credit_utilization,months_with_company,default
0,early_career_urban,3283.287284,0.453151,2.0,0.574281,21.0,0
1,platform_worker,1976.088070,0.597144,0.0,0.855863,39.0,1
2,platform_worker,3299.177450,0.664904,4.0,0.969859,27.0,1
3,platform_worker,2539.814969,0.505792,0.0,0.939466,7.0,0
4,early_career_urban,1655.521735,0.342233,2.0,0.461253,30.0,1
